# Margate Data Cleaning Pipeline
## City-Specific Data Normalization

This notebook handles **Margate specific** data cleaning and normalization:
- Loads raw JSON files from `results_folder/margate/raw_json/`
- Extracts tables from JSON chunks with standard header handling (row 0)
- Applies Margate-specific field mappings
- Outputs clean, normalized data ready for financial analysis

**Input**: Raw JSON files from extraction pipeline  
**Output**: Clean, normalized CSV ready for joining with other cities

## Environment Setup

In [9]:
import pandas as pd
import json
from pathlib import Path
from io import StringIO
import sys
import re

# Add parent directory to path for imports
sys.path.append(str(Path.cwd().parent))

# Import functions directly to avoid relative import issues
try:
    from normalizers import normalize_margate
    print("Successfully imported normalize_margate")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Let's import the functions we need directly...")
    
    # If normalizers import fails, we'll define a simple version here
    def normalize_margate(df, city, source_file=None):
        """Simple Margate normalizer for parsed data"""
        rename_map = {
            "CASE NUMBER": "violation_id_raw",
            "STATUS": "case_status_raw"
        }
        df2 = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}).copy()
        df2["city"] = city
        df2["source_file"] = source_file
        return df2

# Path masking function
def mask_path(p: str | Path) -> str:
    p = Path(p)
    parts = p.parts
    return str(Path(*(["..."] + list(parts[-3:]))))

# Project paths
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[2]
else:
    cwd = Path.cwd()
    if cwd.name == "cleaning":
        ROOT = cwd.parents[1]
    elif cwd.name == "src":
        ROOT = cwd.parent
    else:
        ROOT = cwd

RESULTS_DIR = ROOT / "results_folder"
MARGATE_DIR = RESULTS_DIR / "margate"
CLEAN_DIR = ROOT / "clean_data"
CLEAN_DIR.mkdir(exist_ok=True)

print("ROOT:", mask_path(ROOT))
print("MARGATE_DIR:", mask_path(MARGATE_DIR))
print("CLEAN_DIR:", mask_path(CLEAN_DIR))

# Find JSON files
json_dir = MARGATE_DIR / "raw_json"
if json_dir.exists():
    json_files = list(json_dir.glob("*.json"))
    print(f"Found {len(json_files)} JSON files")
    for f in json_files:
        print(f"  - {f.name}")
else:
    print(f"❌ JSON directory not found: {mask_path(json_dir)}")
    json_files = []

def parse_margate_combined_columns(df):
    """
    Parse Margate's combined columns into separate fields.
    Data is space-separated, not newline-separated.
    
    Combined columns:
    - CASE TYPE ADDRESS: violation_description + space + address
    - DATE OPENED DAYS ACTIVE: date + space + days_number  
    - LAST ACTION NEXT ACTION: action_words + space + next_words
    - RESULT DATE DUE DATE: result_date + space + due_date (or just closed_date)
    """
    print("Parsing Margate combined columns...")
    
    parsed_df = df.copy()
    
    # Parse CASE TYPE ADDRESS - improved logic to handle descriptions starting with numbers
    if "CASE TYPE ADDRESS" in df.columns:
        col_data = df["CASE TYPE ADDRESS"].fillna("").astype(str)
        
        parsed_df["violation_description_raw"] = ""
        parsed_df["address_raw"] = ""
        
        for idx, text in col_data.items():
            if text and text != "nan":
                # Method 1: Look for complete address pattern at end (most reliable)
                address_match = re.search(r'\b(\d+\s+(?:[NSEW]\s+)?[A-Z\s]+(?:RD|ST|AVE|DR|BLVD|LN|CT|PL|WAY|CIR|PKWY)(?:\s+#?\s*\d+)?)\s*$', text, re.IGNORECASE)
                
                if address_match:
                    address = address_match.group(1).strip()
                    violation = text[:address_match.start()].strip()
                    parsed_df.loc[idx, "violation_description_raw"] = violation
                    parsed_df.loc[idx, "address_raw"] = address
                else:
                    # Method 2: Look for address patterns that aren't at the beginning
                    # Find all potential address patterns (number + street name + suffix)
                    address_patterns = re.finditer(r'\b(\d+\s+(?:[NSEW]\s+)?[A-Z\s]*(?:RD|ST|AVE|DR|BLVD|LN|CT|PL|WAY|CIR|PKWY|TER)(?:\s+#?\s*\d+)?)\b', text, re.IGNORECASE)
                    
                    # Convert to list to work with
                    matches = list(address_patterns)
                    
                    if matches:
                        # Take the last (rightmost) address pattern found
                        last_match = matches[-1]
                        address = last_match.group(1).strip()
                        violation = text[:last_match.start()].strip()
                        parsed_df.loc[idx, "violation_description_raw"] = violation
                        parsed_df.loc[idx, "address_raw"] = address
                    else:
                        # Method 3: Fallback - look for patterns that suggest address vs description
                        words = text.split()
                        address_start = -1
                        
                        # Skip the first few words if they look like violation descriptions
                        # Look for street indicators further in the text
                        for i in range(len(words)):
                            word = words[i]
                            # If this word starts with a digit AND we're not at the very beginning
                            # AND the next few words contain street indicators
                            if re.match(r'^\d+$', word) and i > 0:
                                # Check if following words contain street suffixes
                                remaining_text = " ".join(words[i:])
                                if re.search(r'\b(?:RD|ST|AVE|DR|BLVD|LN|CT|PL|WAY|CIR|PKWY|TER)\b', remaining_text, re.IGNORECASE):
                                    address_start = i
                                    break
                        
                        if address_start > 0:
                            violation = " ".join(words[:address_start])
                            address = " ".join(words[address_start:])
                            parsed_df.loc[idx, "violation_description_raw"] = violation
                            parsed_df.loc[idx, "address_raw"] = address
                        else:
                            # No clear separation found - keep as violation description only
                            parsed_df.loc[idx, "violation_description_raw"] = text
                            parsed_df.loc[idx, "address_raw"] = ""
        
        print("  Parsed CASE TYPE ADDRESS into violation_description_raw + address_raw")
    
    # Parse DATE OPENED DAYS ACTIVE
    if "DATE OPENED DAYS ACTIVE" in df.columns:
        col_data = df["DATE OPENED DAYS ACTIVE"].fillna("").astype(str)
        
        parsed_df["opened_date_raw"] = ""
        parsed_df["days_active_raw"] = ""
        
        for idx, text in col_data.items():
            if text and text != "nan":
                # Look for date pattern followed by number
                date_match = re.search(r'(\d{1,2}/\d{1,2}/\d{2,4})', text)
                if date_match:
                    date_part = date_match.group(1)
                    remaining = text[date_match.end():].strip()
                    # Extract number from remaining
                    number_match = re.search(r'\d+', remaining)
                    if number_match:
                        parsed_df.loc[idx, "opened_date_raw"] = date_part
                        parsed_df.loc[idx, "days_active_raw"] = number_match.group()
                    else:
                        parsed_df.loc[idx, "opened_date_raw"] = date_part
        
        print("  Parsed DATE OPENED DAYS ACTIVE into opened_date_raw + days_active_raw")
    
    # Parse LAST ACTION NEXT ACTION - improved logic
    if "LAST ACTION NEXT ACTION" in df.columns:
        col_data = df["LAST ACTION NEXT ACTION"].fillna("").astype(str)
        
        parsed_df["last_action_raw"] = ""
        parsed_df["next_action_raw"] = ""
        parsed_df["closed_date_raw"] = ""
        
        for idx, text in col_data.items():
            if text and text != "nan":
                # Check for CASE CLOSED pattern first
                if "CASE CLOSED" in text.upper():
                    parsed_df.loc[idx, "last_action_raw"] = text
                    # Extract date from CASE CLOSED line
                    date_match = re.search(r'\d{1,2}/\d{1,2}/\d{2,4}', text)
                    if date_match:
                        parsed_df.loc[idx, "closed_date_raw"] = date_match.group()
                else:
                    # Improved action parsing
                    # Look for common patterns where actions split
                    
                    # Pattern 1: "INITIAL INSPECTION" (no split needed)
                    if text.strip() in ["INITIAL INSPECTION", "REINSPECTION", "COMPLIANCE INSPECTION"]:
                        parsed_df.loc[idx, "last_action_raw"] = text.strip()
                        parsed_df.loc[idx, "next_action_raw"] = ""
                    
                    # Pattern 2: "NOTICE OF VIOLATION-MAIL REINSPECTION"
                    elif "NOTICE OF VIOLATION" in text and "REINSPECTION" in text:
                        if text.endswith("REINSPECTION"):
                            # Split at the word before REINSPECTION
                            parts = text.split()
                            if "REINSPECTION" in parts:
                                reinsp_idx = parts.index("REINSPECTION")
                                last_action = " ".join(parts[:reinsp_idx])
                                next_action = " ".join(parts[reinsp_idx:])
                                parsed_df.loc[idx, "last_action_raw"] = last_action
                                parsed_df.loc[idx, "next_action_raw"] = next_action
                            else:
                                parsed_df.loc[idx, "last_action_raw"] = text
                        else:
                            parsed_df.loc[idx, "last_action_raw"] = text
                    
                    # Pattern 3: Other combinations
                    elif "RED TAG" in text:
                        if text.endswith("RED TAG"):
                            parts = text.split()
                            if "RED" in parts and "TAG" in parts:
                                red_idx = parts.index("RED")
                                last_action = " ".join(parts[:red_idx])
                                next_action = " ".join(parts[red_idx:])
                                parsed_df.loc[idx, "last_action_raw"] = last_action
                                parsed_df.loc[idx, "next_action_raw"] = next_action
                            else:
                                parsed_df.loc[idx, "last_action_raw"] = text
                        else:
                            parsed_df.loc[idx, "last_action_raw"] = text
                    
                    else:
                        # Default: keep as single action
                        parsed_df.loc[idx, "last_action_raw"] = text
                        parsed_df.loc[idx, "next_action_raw"] = ""
        
        print("  Parsed LAST ACTION NEXT ACTION with improved logic")
    
    # Parse RESULT DATE DUE DATE
    if "RESULT DATE DUE DATE" in df.columns:
        col_data = df["RESULT DATE DUE DATE"].fillna("").astype(str)
        
        parsed_df["result_date_raw"] = ""
        parsed_df["due_date_raw"] = ""
        # closed_date_raw already initialized above
        if "closed_date_raw" not in parsed_df.columns:
            parsed_df["closed_date_raw"] = ""
        
        for idx, text in col_data.items():
            if text and text != "nan":
                # Find all dates in the text
                dates = re.findall(r'\d{1,2}/\d{1,2}/\d{2,4}', text)
                if len(dates) == 2:
                    # Two dates: result and due
                    parsed_df.loc[idx, "result_date_raw"] = dates[0]
                    parsed_df.loc[idx, "due_date_raw"] = dates[1]
                elif len(dates) == 1:
                    # One date: could be closed date if it appears to be a single date
                    if text.count('/') <= 2:  # Single date format
                        parsed_df.loc[idx, "closed_date_raw"] = dates[0]
                    else:
                        parsed_df.loc[idx, "result_date_raw"] = dates[0]
        
        print("  Parsed RESULT DATE DUE DATE into result_date_raw + due_date_raw + closed_date_raw")
    
    # Remove original combined columns
    cols_to_drop = ["CASE TYPE ADDRESS", "DATE OPENED DAYS ACTIVE", "LAST ACTION NEXT ACTION", "RESULT DATE DUE DATE"]
    parsed_df = parsed_df.drop(columns=[col for col in cols_to_drop if col in parsed_df.columns])
    
    print(f"Parsing complete! New columns: {list(parsed_df.columns)}")
    return parsed_df

❌ Import error: attempted relative import with no known parent package
Let's import the functions we need directly...
ROOT: ...\Edilma Projects\LandingAI-Hack\coderisk-sf
MARGATE_DIR: ...\coderisk-sf\results_folder\margate
CLEAN_DIR: ...\LandingAI-Hack\coderisk-sf\clean_data
Found 1 JSON files
  - Margate_CodeViolations-To-Feb-2024 Building Dept.json


## Extract Tables from JSON Chunks

In [10]:
def extract_margate_tables(json_file_path):
    """
    Extract tables from Margate JSON chunks with standard header handling.
    Each table has headers in row 0.
    """
    print(f"Processing Margate JSON: {Path(json_file_path).name}")
    
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_tables = []
    
    if 'chunks' in data:
        for i, chunk in enumerate(data['chunks']):
            if chunk.get('type') == 'table' and 'markdown' in chunk:
                print(f"  Processing table chunk {i+1}")
                
                # Parse HTML table with standard header handling
                df = parse_html_table(chunk['markdown'])
                if df is not None and not df.empty:
                    # Add metadata with masked path
                    df['chunk_id'] = i
                    df['source_file'] = mask_path(json_file_path)
                    all_tables.append(df)
                    print(f"    Extracted {len(df)} rows")
    
    if all_tables:
        combined_df = pd.concat(all_tables, ignore_index=True)
        print(f"Total extracted: {len(combined_df)} rows")
        return combined_df
    else:
        print("❌ No table chunks found")
        return pd.DataFrame()

def parse_html_table(html_content):
    """Parse HTML table with standard header detection for Margate data"""
    try:
        if '<table' in html_content.lower():
            # Parse table without assuming header location
            tables = pd.read_html(StringIO(html_content), header=None)
            if tables:
                df = tables[0]
                
                # Use first row as column names and remove it from data
                if len(df) > 1:
                    headers = df.iloc[0].fillna('').astype(str).tolist()
                    data_df = df.iloc[1:].reset_index(drop=True)
                    data_df.columns = headers[:len(data_df.columns)]
                    return data_df
                else:
                    return df
    except Exception as e:
        print(f"    ❌ Parse error: {e}")
    return None

## Process All Margate Files

In [11]:
# Process all Margate JSON files
print("Processing ALL Margate files...")

all_data = []

for i, json_file in enumerate(json_files, 1):
    print(f"\n[{i}/{len(json_files)}] Processing: {json_file.name}")
    df = extract_margate_tables(json_file)
    if not df.empty:
        all_data.append(df)
        print(f"  Got {len(df)} rows")
    else:
        print(f"  ❌ No data from this file")

if all_data:
    # Combine all data
    margate_raw = pd.concat(all_data, ignore_index=True)
    
    print(f"\nRAW MARGATE DATA:")
    print(f"Shape: {margate_raw.shape}")
    print(f"Columns: {list(margate_raw.columns)}")
    print(f"\nSample data:")
    print(margate_raw.head(3))
    
else:
    print("❌ No data extracted from any files")
    margate_raw = pd.DataFrame()

Processing ALL Margate files...

[1/1] Processing: Margate_CodeViolations-To-Feb-2024 Building Dept.json
Processing Margate JSON: Margate_CodeViolations-To-Feb-2024 Building Dept.json
  Processing table chunk 4
    Extracted 17 rows
  Processing table chunk 8
    Extracted 17 rows
  Processing table chunk 12
    Extracted 17 rows
  Processing table chunk 16
    Extracted 17 rows
  Processing table chunk 20
    Extracted 17 rows
  Processing table chunk 24
    Extracted 17 rows
  Processing table chunk 28
    Extracted 17 rows
  Processing table chunk 32
    Extracted 17 rows
  Processing table chunk 36
    Extracted 17 rows
  Processing table chunk 40
    Extracted 17 rows
  Processing table chunk 44
    Extracted 17 rows
  Processing table chunk 48
    Extracted 17 rows
  Processing table chunk 52
    Extracted 17 rows
  Processing table chunk 56
    Extracted 17 rows
  Processing table chunk 60
    Extracted 17 rows
  Processing table chunk 64
    Extracted 17 rows
  Processing table

## Data Cleaning

In [12]:
if not margate_raw.empty:
    print("Cleaning Margate data...")
    
    # Step 1: Parse combined columns into separate fields
    print("\nStep 1: Parsing combined columns...")
    margate_parsed = parse_margate_combined_columns(margate_raw)
    
    print(f"After parsing: {margate_parsed.shape}")
    print(f"New columns: {list(margate_parsed.columns)}")
    
    # Step 2: Standard cleaning
    print(f"\nStep 2: Standard data cleaning...")
    margate_clean = margate_parsed.copy()
    
    print(f"Before cleaning: {len(margate_clean)} rows")
    
    # Remove any obvious header repeats using CASE NUMBER column
    if "CASE NUMBER" in margate_clean.columns:
        header_mask = margate_clean["CASE NUMBER"].astype(str).str.strip().str.upper() == "CASE NUMBER"
        rows_before = len(margate_clean)
        margate_clean = margate_clean[~header_mask]
        print(f"Removed {rows_before - len(margate_clean)} header repeat rows")
    
    # Remove empty rows
    rows_before = len(margate_clean)
    margate_clean = margate_clean.dropna(how='all')
    print(f"Removed {rows_before - len(margate_clean)} completely empty rows")
    
    print(f"After cleaning: {len(margate_clean)} rows")
    
    # Show data overview
    print(f"\nColumn overview:")
    for col in margate_clean.columns:
        non_null = margate_clean[col].notna().sum()
        print(f"  {col}: {non_null} non-null values")
    
    print(f"\nSample cleaned and parsed data:")
    display_cols = ["CASE NUMBER", "violation_description_raw", "address_raw", "opened_date_raw", "STATUS"]
    available_cols = [col for col in display_cols if col in margate_clean.columns]
    print(margate_clean[available_cols].head(3))
    
else:
    print("❌ No raw data to clean")
    margate_clean = pd.DataFrame()

Cleaning Margate data...

Step 1: Parsing combined columns...
Parsing Margate combined columns...
  Parsed CASE TYPE ADDRESS into violation_description_raw + address_raw
  Parsed DATE OPENED DAYS ACTIVE into opened_date_raw + days_active_raw
  Parsed CASE TYPE ADDRESS into violation_description_raw + address_raw
  Parsed DATE OPENED DAYS ACTIVE into opened_date_raw + days_active_raw
  Parsed LAST ACTION NEXT ACTION with improved logic
  Parsed RESULT DATE DUE DATE into result_date_raw + due_date_raw + closed_date_raw
Parsing complete! New columns: ['CASE NUMBER', 'STATUS', 'chunk_id', 'source_file', '', 'violation_description_raw', 'address_raw', 'opened_date_raw', 'days_active_raw', 'last_action_raw', 'next_action_raw', 'closed_date_raw', 'result_date_raw', 'due_date_raw']
After parsing: (741, 14)
New columns: ['CASE NUMBER', 'STATUS', 'chunk_id', 'source_file', '', 'violation_description_raw', 'address_raw', 'opened_date_raw', 'days_active_raw', 'last_action_raw', 'next_action_raw', 

## Apply Normalization

In [13]:
if not margate_clean.empty:
    print("Applying Margate normalization...")
    
    # Apply the normalize_margate function
    margate_normalized = normalize_margate(
        margate_clean, 
        city="Margate", 
        source_file="margate_data.json"
    )
    
    print(f"Normalization complete!")
    print(f"Shape: {margate_normalized.shape}")
    print(f"Columns: {list(margate_normalized.columns)}")
    
    # Show sample normalized data
    print(f"\nSample normalized data:")
    key_cols = ['violation_id_raw', 'address_raw', 'case_status_raw', 'opened_date_raw']
    display_cols = [col for col in key_cols if col in margate_normalized.columns]
    print(margate_normalized[display_cols].head(5))
    
    # Show unique case count
    if 'violation_id_raw' in margate_normalized.columns:
        unique_cases = margate_normalized['violation_id_raw'].nunique()
        print(f"\nFound {unique_cases} unique cases")
    
else:
    print("❌ No clean data to normalize")
    margate_normalized = pd.DataFrame()

Applying Margate normalization...
Normalization complete!
Shape: (741, 15)
Columns: ['violation_id_raw', 'case_status_raw', 'chunk_id', 'source_file', '', 'violation_description_raw', 'address_raw', 'opened_date_raw', 'days_active_raw', 'last_action_raw', 'next_action_raw', 'closed_date_raw', 'result_date_raw', 'due_date_raw', 'city']

Sample normalized data:
  violation_id_raw           address_raw case_status_raw opened_date_raw
0      23-00100401  5301 W ATLANTIC BLVD          ACTIVE        11/07/23
1      23-00100402  5301 W ATLANTIC BLVD          ACTIVE        11/07/23
2      24-00100048  5375 W ATLANTIC BLVD          ACTIVE         2/07/24
3      23-00100419  5499 W ATLANTIC BLVD          ACTIVE        11/14/23
4      23-00100039  5642 W ATLANTIC BLVD          ACTIVE         2/23/23

Found 741 unique cases


## Save Results

In [14]:
if not margate_normalized.empty:
    # Save cleaned data
    output_file = CLEAN_DIR / "margate_clean.csv"
    margate_normalized.to_csv(output_file, index=False)
    
    print(f"Saved to: {mask_path(output_file)}")
    
    # Final stats
    print(f"\nMARGATE RESULTS:")
    print(f"  - Total rows: {len(margate_normalized)}")
    if 'violation_id_raw' in margate_normalized.columns:
        print(f"  - Unique cases: {margate_normalized['violation_id_raw'].nunique()}")
    if 'case_status_raw' in margate_normalized.columns:
        print(f"  - Status breakdown:")
        status_counts = margate_normalized['case_status_raw'].value_counts()
        for status, count in status_counts.head(5).items():
            print(f"    {status}: {count}")
    
    # Date range if available (with format specification to avoid warning)
    if 'opened_date_raw' in margate_normalized.columns:
        # Try common date formats first
        date_col = None
        for fmt in ['%m/%d/%Y', '%m/%d/%y', '%Y-%m-%d']:
            try:
                date_col = pd.to_datetime(margate_normalized['opened_date_raw'], format=fmt, errors='coerce')
                break
            except:
                continue
        
        # Fallback to dateutil parsing if format detection fails
        if date_col is None:
            date_col = pd.to_datetime(margate_normalized['opened_date_raw'], errors='coerce')
        
        valid_dates = date_col.dropna()
        if not valid_dates.empty:
            print(f"  - Date range: {valid_dates.min().date()} to {valid_dates.max().date()}")
    
    print(f"\nSUCCESS! Margate data ready for financial analysis!")
    
else:
    print("❌ No data to save")

Saved to: ...\coderisk-sf\clean_data\margate_clean.csv

MARGATE RESULTS:
  - Total rows: 741
  - Unique cases: 741
  - Status breakdown:
    ACTIVE: 494
    CASE CLOSED: 239
    IN COMPLIANCE/OPEN FINES: 7
    VOIDED: 1

SUCCESS! Margate data ready for financial analysis!
